In [ ]:
# @title Install necessary libraries
!pip install -q google-genai google-cloud-storage pypdf google-colab
print("Libraries installed.")

Libraries installed.


In [ ]:
# @title Import libraries and authenticate
from google import genai
from google.cloud import storage
from google.colab import auth
from google.genai.types import Part, GenerateContentConfig # Import Part for file input
import pypdf
import os
import json
import textwrap # For printing long text nicely
from IPython.display import display, Markdown # For better output formatting

# Authenticate the Colab user for GCP access (like GCS)
print("Authenticating user for Google Cloud access...")
auth.authenticate_user()
print("Authentication successful.")

Authenticating user for Google Cloud access...
Authentication successful.


In [ ]:
!gcloud auth list


       Credentialed Accounts
ACTIVE  ACCOUNT
*       KalpashreeRajanna@gmail.com

To set the active account, run:
    $ gcloud config set account `ACCOUNT`



In [ ]:
!gcloud projects list


PROJECT_ID                  NAME                        PROJECT_NUMBER
extraction-papers-talhouk2  extraction-papers-talhouk2  893111543543
pro-tuner-466721-j1         My First Project            1012279005356
resounding-keel-466721-r8   My First Project            378047208130
spatial-racer-466721-k4     My First Project            706052804423
tensile-talent-466721-p7    paper-extraction-talhouk    928325513656


In [ ]:
!gcloud projects add-iam-policy-binding paper-extraction-talhouk \
  --member="user:kalpashreerajanna@gmail.com" \
  --role="roles/aiplatform.user"


ERROR: (gcloud.projects.add-iam-policy-binding) [KalpashreeRajanna@gmail.com] does not have permission to access projects instance [paper-extraction-talhouk:getIamPolicy] (or it may not exist): The caller does not have permission. This command is authenticated as KalpashreeRajanna@gmail.com which is the active account specified by the [core/account] property


In [ ]:
!gcloud projects get-iam-policy paper-extraction-talhouk --format=json > policy.json


ERROR: (gcloud.projects.get-iam-policy) [KalpashreeRajanna@gmail.com] does not have permission to access projects instance [paper-extraction-talhouk:getIamPolicy] (or it may not exist): The caller does not have permission. This command is authenticated as KalpashreeRajanna@gmail.com which is the active account specified by the [core/account] property


In [ ]:
# @title Configure Google Cloud Settings and connect Google Cloud Storage
#    Replace with your actual Project ID, Bucket Name, and Prefix (folder)
GCP_PROJECT_ID = "extraction-papers-talhouk2"
GCS_BUCKET_NAME = "tlab-extractionbucket3"
GCS_PREFIX = "papers3" # e.g., "papers/neuroscience/" or "" for root
LOCATION = "us-central1" # location you want to use for data procssing

# Optional: Set project for GCS client (often inferred after auth, but explicit is safer)
try:
    storage_client = storage.Client(project=GCP_PROJECT_ID)
    print(f"Google Cloud Storage client initialized for project '{GCP_PROJECT_ID}'.")
except Exception as e:
     print(f"Error initializing GCS Client: {e}. Trying without explicit project ID.")
     try:
         storage_client = storage.Client()
         inferred_project = storage_client.project
         print(f"Google Cloud Storage client initialized using inferred project '{inferred_project}'.")
         GCP_PROJECT_ID = inferred_project # Update if inferred
     except Exception as e2:
         print(f"FATAL ERROR: Could not initialize GCS Client: {e2}")
         storage_client = None # Ensure it's None if failed

Google Cloud Storage client initialized for project 'extraction-papers-talhouk2'.


In [ ]:
# # @title Model Selection
MODEL_NAME = "gemini-2.0-flash-001"
client = genai.Client(vertexai=True, project=GCP_PROJECT_ID, location=LOCATION)
print(f"Using Gemini model: {MODEL_NAME}")

Using Gemini model: gemini-2.0-flash-001


In [ ]:
# @title Model Selection - USE THIS ONE
MODEL_NAME = "gemini-2.0-flash-001"
client = genai.Client(api_key='')
print(f"Using Gemini model: {MODEL_NAME}")

Using Gemini model: gemini-2.0-flash-001


In [ ]:
# @title Define Entities To Extract
ENTITIES_TO_EXTRACT = {
 # "Title of Paper": "Extract the title of the paper, e.g., POLE exonuclease domain mutation predicts long progression-free survival in grade 3 endometrioid carcinoma of the endometrium",
    #"Author list": "Extract the full list of authors, e.g., DeRycke M, Andersen J.D., Harrington K.M., Pambuccian S.E., Kalloger S.E., Boylan K.L.M., et al.",
    #"Histology": "Based on the primary cancer type under study, classify the paper as one of the following: Ovarian (including ovarian cancers with endometrioid histology or arising from endometriosis, such as endometriosis-associated ovarian cancer), Endometrial (only if the primary cancer is located in the endometrium), Both (if both ovarian and endometrial cancers are studied as distinct primary cancers), Neither (if neither is a focus of the study), Do not classify as 'Endometrial' just because of endometrioid histology unless the primary site is explicitly the uterus. Likewise, do not choose 'Both' unless both cancers are discussed as distinct primary diseases.”",

 #   "Histology subtype":"""Extract up to 5 distinct histological cancer subtypes discussed in the study, focusing on the most prominently analyzed or referenced.


# Use the following controlled vocabulary only (do not invent new labels):

# Endometrial Cancer

# Endometrioid Endometrial Cancer

# Non-Endometrioid Endometrial Cancer

# Endometrial Serous Carcinoma

# Endometrial Clear Cell Carcinoma

# Uterine Papillary Serous Carcinoma

# Endometrial Mucinous Carcinoma

# Small Cell Carcinoma of the Endometrium

# Malignant Mixed Mesodermal Tumour (carcinosarcoma)

# Not otherwise specified Endometrial Adenocarcinoma

# Endometrioid Ovarian Carcinoma

# Clear Cell Ovarian Carcinoma

# Endometriosis-associated Ovarian Cancer

# Serous Ovarian Carcinoma

# High-Grade Serous Ovarian Carcinoma

# Important:

# Do not confuse Endometrioid Ovarian Carcinoma with Endometriosis-associated Ovarian Cancer — they are distinct subtypes.

# List only the most frequently or centrally analyzed subtypes.

# If more than 5 are mentioned, return the top 5 and add "Other" at the end.

# Format your answer as a comma-separated list, e.g.:
# Endometrioid Endometrial Cancer, High-Grade Serous Ovarian Carcinoma, Endometrial Serous Carcinoma, Clear Cell Ovarian Carcinoma, Other""",
    #  "Publisher":"Extract the publisher, e.g., Gynecol Oncol",
  # "N of total cohort":"Extract the full sample size of the patient cohort that was analysed in the study. Carefully evaluate if multiple cohorts are mentioned and add them up. Return only one number, e.g., 192",
   # "Date of diagnosis":"Extract the date range of when the patients in the cohort received the first diagnosis, e.g., 2005 - 2011",
    # "Primary location of study":"Extract the primary city and country of the study, e.g,. Vancouver, Canada",
    # "Treatments applied in the study": "Extract a detailed summary of all treatments applied to patients in the study. Include initial treatments (e.g., surgery, debulking, salpingo-oophorectomy), chemotherapy (specify neoadjuvant or adjuvant, and agents like platinum-based, taxol, cisplatin), radiation therapy (e.g., external beam, vaginal brachytherapy), hormonal therapy (e.g., megestrol, progesterone IUS), and any targeted or immunotherapies. For each treatment, report the number of patients and percentages if available. State clearly if patients received no treatment. If treatments differ by cancer subtype, report separately. If only general treatment statements are given (e.g., 'all received platinum-based chemotherapy after surgery'), extract those directly. If treatment information is not available, return 'N/A'. Return answers in a structured, itemized format with precise counts and treatment types.",
    # "Analysis":"Extract the technology used to perform the analysis of the primary samples, e.g., IHC TMA",
    "Analysed Biomarkers":"Extract which biomarkers were analysed in the study, e.g., POLE, p53, MMR, L1CAM, BRCA1. Only extract the top 3 most significant biomarkers, often mentioned in the methods and/or abstract and/or conclusion",
    # "Significant in multivariate (1), univariate (2), or both (3), analysis not significant (4) or unknown (5)":"Determine whether the biomarker or treatment of interest showed statistical significance in univariate analysis, multivariate analysis, both, neither, or if unknown/not reported (codes: 1 = multivariate only, 2 = univariate only, 3 = both, 4 = neither, 5 = unknown), where univariate analysis examines one variable at a time and multivariate analysis adjusts for multiple variables (often described using terms like “Cox regression,” “adjusted for,” or “independent predictor”); if statistical significance is reported without mention of adjustment or multivariate methods, assume univariate analysis.",
    # "Did the identified biomarker or treatment improve prognosis? Prognosis: Improved (1), Worse (2), Both (3), not significant (4), unknown (5)":"Determine the overall effect of the identified biomarker(s) or treatment on patient prognosis and return only one code—1 = improved, 2 = worse, 3 = both (if different subgroups or biomarkers show opposing effects), 4 = no significant effect, 5 = unknown—and if there are mixed effects, return 3 with a brief rationale.",
    # "N histotype":"Extract and explicitly state the histotype composition of the full study cohort. Only report histotypes, such as: Endometrial Carcinoma, Endometrioid Endometrial Carcinoma, Endometrioid Ovarian Carcinoma, Clear Cell Ovarian Carcinoma, Serous Ovarian Carcinoma, etc. Provide exact numbers for each histotype if available. If only percentages are given, calculate and report estimated numbers based on total cohort size. Do not include molecular subtypes, treatment information, or any other clinical characteristics. If no histotype breakdown is provided, return 'N/A'. The output should be limited strictly to histotype cohort composition.",
    # "Was a recent histological review (using 2014 WHO criteria) performed?":"The 2014 WHO criteria specify that the study should provide detailed diagnostic criteria, including histological features, grading and immunohistochemical markers; Or whether there were one or more pathologists who reviewed cases. Determine if this information is available and respond with Yes or No",
    # "5-Year Survival rates (overall survival unless states otherwise)":"Extract the 5-year survival rate, prioritizing overall survival (OS). If OS is not explicitly stated in the main text, check tables, figures, or Kaplan-Meier plots. If only a different survival type is available (e.g., progression-free survival [PFS] or disease-free survival [DFS]), report it and clearly specify the type. Format the result as: 'OVERALL SURVIVAL: 88.3%' or 'PFS: 76.1%'. If no survival data is found anywhere in the paper, return: 'NOT REPORTED'.",
    # "Mean age of participants":"Extract the mean age of participants. If mean is not available, provide the median age. Always specify which is used. Format: MEAN: 42.9, MEDIAN: 45.3. If neither is found, return: AGE NOT REPORTED",
    #"% of participants with clinicopathological features (Stage I or II)":"Extract the percentage of study participants with disease Stage I or Stage II tumors, e.g., 97%",
    # "REMARK Critera":"Check if the study explicitly states that it followed REMARK criteria. Look for phrases such as:'according to the REMARK criteria','REMARK reporting criteria','in accordance with REMARK guidelines','REMARK recommendations', or any mention of 'REMARK' in reference to study design or analysis. If such a phrase is found, return: 'Yes', If there is no mention of REMARK criteria or guidelines, return: 'No', Do not infer based on formatting or study structure — rely only on explicit textual mention.",
    # "Conclusion of the study":"Extract a brief summary of the conclusion of the study, e.g., ProMisE subgroups are associated with patient outcome in both univariate and multivariate analysis."

}

In [ ]:
# @title Helper Functions
import re

def list_pdfs_in_gcs(client, bucket_name, prefix):
    """Lists PDF files in a GCS bucket/prefix."""
    if not client:
        print("GCS client not initialized. Cannot list files.")
        return []
    print(f"Listing PDFs in gs://{bucket_name}/{prefix}...")
    try:
        bucket = client.get_bucket(bucket_name)
        blobs = client.list_blobs(bucket, prefix=prefix)
        pdf_files = [f"gs://{bucket_name}/{blob.name}" for blob in blobs if blob.name.lower().endswith(".pdf") and blob.size > 0]
        print(f"Found {len(pdf_files)} PDF files.")
        return pdf_files
    except Exception as e:
        print(f"Error listing GCS files: {e}")
        return []

def extract_entities_with_gemini_uri(gcs_uri, entities):
    """Uses Gemini to extract specified entities directly from a PDF URI."""
    if not gcs_uri or not gcs_uri.startswith("gs://"):
        return {"error": "Invalid GCS URI provided"}

    print(f"  Preparing request for Gemini with URI: {gcs_uri}")
    # Create a Part object representing the PDF file in GCS
    try:
        pdf_file = Part.from_uri(
            file_uri=gcs_uri,
            mime_type="application/pdf" # Crucial for the model to know the file type
        )
    except Exception as e:
        print(f"  Error creating Part from URI {gcs_uri}: {e}")
        return {"error": f"Failed to create Part from URI: {e}"}

    system_instruction=(
        """Analyze the provided PDF file (research paper) and extract the entities given to you in your prompt.
        Provide the output strictly in JSON format with the keys from the prompt.
        If an entity cannot be found or is not applicable, use the value "n/a".
        JSON Output:"""
    ),

    # Construct the prompt, instructing the model to analyze the *provided file*
    print(f"  Calling Gemini API for {gcs_uri}...")
    results = {}

    for entity in entities:
        print(f"  Processing entity: {entity}")
        prompt = f"""
        Extract the following entity from the research paper:

        **Entity**: {entity}

        **Instructions**: {entities[entity]}

        Please return the result strictly in the following JSON format:
        {{
          "{entity}": {{
            "value": "The extracted information",
            "confidence": ["confidence score from 0 to 100 as a percentage", a short one sentence rationale for why the confidence score was given]
          }}
        }}

        If the entity is not found, set "value" to "n/a" and "confidence" to 0.
        """

        # Make the API call with both the prompt and the file Part
        try:
            # The request contains both the text prompt and the file reference
            response = client.models.generate_content(
                model=MODEL_NAME,
                config=GenerateContentConfig(
                    system_instruction=system_instruction),
                contents=[prompt, pdf_file],
                )
            print(f"  Received response from Gemini.")

            # Basic check for response content
            if not response.candidates or not response.candidates[0].content.parts:
                print("  Warning: Gemini response seems empty or malformed.")
                # Check for safety ratings or finish reasons if available
                try:
                    finish_reason = response.candidates[0].finish_reason
                    safety_ratings = response.candidates[0].safety_ratings
                    print(f"  Finish Reason: {finish_reason}")
                    print(f"  Safety Ratings: {safety_ratings}")
                    if response.prompt_feedback:
                        print(f"  Prompt Feedback: {response.prompt_feedback}")
                except (AttributeError, IndexError):
                    pass # Ignore if these attributes don't exist
                return {"error": "Empty or malformed response from API", "raw_response": str(response)}

            extracted_text = response.text
            #print(f"Gemini returned {response.text}")

            # Attempt to parse the JSON response
            try:
                # Clean potential markdown formatting around the JSON
                if extracted_text.strip().startswith("```json"):
                    extracted_text = extracted_text.strip()[7:-3].strip()
                elif extracted_text.strip().startswith("```"):
                    extracted_text = extracted_text.strip()[3:-3].strip()

                result = json.loads(extracted_text)
                for entry in result:
                    results[entity] = result[entry]

            except json.JSONDecodeError:
                print(f"  Warning: Failed to parse JSON response from Gemini.")
                # Log the raw response for debugging
                print(f"  Raw response snippet:\n---\n{textwrap.shorten(extracted_text, width=200, placeholder='...')}\n---")
                #return {"error": "Failed to parse JSON response", "raw_response": extracted_text}
            except Exception as e:
                print(f"  Error processing Gemini response content: {e}")
                #return {"error": f"Error processing response content: {e}", "raw_response": extracted_text}

        except Exception as e:
            # Catch potential API errors (e.g., permission denied on GCS URI, rate limits)
            print(f"  ERROR calling Gemini API for {gcs_uri}: {e}")
            # Check if the error message indicates a permission issue
            if "permission denied" in str(e).lower() or "could not access" in str(e).lower():
                print("  This might be a GCS permission issue. See the note in Cell 3 about granting the 'Storage Object Viewer' role.")
                #return {"error": f"Gemini API call failed: {e}"}
      # Validate that all requested keys are present

    validated_result = {}
    for entity in entities:
        validated_result[entity] = results.get(entity, "n/a") # Default to "Not Found" if key is missing
    return validated_result

In [ ]:
# @title ## MEREDITH Extractor

all_results = []
limit = 100

# Check prerequisites
if not client:
    print("ERROR: GenAI Client not initialized. Please fix configuration.")
elif not storage_client:
    print("ERROR: GCS client not initialized. Please fix configuration.")
else:
    # 1. List PDF files in GCS
    pdf_uris = list_pdfs_in_gcs(storage_client, GCS_BUCKET_NAME, GCS_PREFIX)

    if not pdf_uris:
        print("\nNo PDF files found or error listing files. Ensure configuration is correct and files exist.")
    else:
        print(f"\nStarting entity extraction for {len(pdf_uris)} PDF files using URIs...")

        # 2. Loop through each PDF URI
        for i, pdf_uri in enumerate(pdf_uris):
          if i > limit:
            break
          print(f"\n--- Processing file {i+1}/{len(pdf_uris)}: {pdf_uri} ---")
          file_result = {"pdf_uri": pdf_uri}

          # 3. Extract Entities using Gemini directly with the URI
          entities_data = extract_entities_with_gemini_uri(pdf_uri, ENTITIES_TO_EXTRACT)
          file_result.update(entities_data)

          if "error" in entities_data:
              print(f"Extraction failed for {pdf_uri}: {entities_data['error']}")
          else:
                print(f"Extraction successful for {pdf_uri}.")

          all_results.append(file_result)

          # Optional: Add a small delay to avoid hitting potential rate limits
          # time.sleep(1) # Sleep for 1 second between requests

        print("\n--- Processing Complete ---")


print(f"\n--- Extracted Information ({len(all_results)} files processed) ---")

# Check if any results were generated
if not all_results:
    print("No results were generated. Please check previous cell outputs for errors.")
else:
    # Iterate and display results for each file
    for result in all_results:
        display(Markdown(f"**File:** `{result.get('pdf_uri', 'N/A')}`")) # Get URI safely
        if "error" in result:
            # Display error message clearly
            display(Markdown(f"  - **Status:** <font color='red'>Error - {result['error']}</font>"))
            # Optionally display raw response snippet if available and useful
            if "raw_response" in result:
                 display(Markdown(f"  - **Raw Response Snippet:**"))
                 display(Markdown(f"```\n{textwrap.shorten(result['raw_response'], width=150, placeholder='...')}\n```"))
        else:
            # Display success and extracted entities
            display(Markdown(f"  - **Status:** <font color='green'>Success</font>"))
            for entity in ENTITIES_TO_EXTRACT:
                value = result.get(entity, "Not Found") # Use .get for safety
                # Format list display (e.g., for authors)
                if isinstance(value, list):
                     display(Markdown(f"  - **{entity}:** {', '.join(map(str, value)) if value else 'None'}")) # Handle non-string list items
                else:
                     display(Markdown(f"  - **{entity}:** {value}"))
        display(Markdown("---")) # Add a separator between file results


Listing PDFs in gs://tlab-extractionbucket3/papers3...
Found 129 PDF files.

Starting entity extraction for 129 PDF files using URIs...

--- Processing file 1/129: gs://tlab-extractionbucket3/papers3/S023_Frequent Mismatch Repair Protein Deficiency in Mixed Endometrioid and Clear Cell Carcinoma of the Endometrium.pdf ---
  Preparing request for Gemini with URI: gs://tlab-extractionbucket3/papers3/S023_Frequent Mismatch Repair Protein Deficiency in Mixed Endometrioid and Clear Cell Carcinoma of the Endometrium.pdf
  Calling Gemini API for gs://tlab-extractionbucket3/papers3/S023_Frequent Mismatch Repair Protein Deficiency in Mixed Endometrioid and Clear Cell Carcinoma of the Endometrium.pdf...
  Processing entity: Analysed Biomarkers
  ERROR calling Gemini API for gs://tlab-extractionbucket3/papers3/S023_Frequent Mismatch Repair Protein Deficiency in Mixed Endometrioid and Clear Cell Carcinoma of the Endometrium.pdf: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Invalid or u

**File:** `gs://tlab-extractionbucket3/papers3/S023_Frequent Mismatch Repair Protein Deficiency in Mixed Endometrioid and Clear Cell Carcinoma of the Endometrium.pdf`

  - **Status:** <font color='green'>Success</font>

  - **Analysed Biomarkers:** n/a

---

...

---

**File:** `gs://tlab-extractionbucket3/papers3/S72_L1 expression as a predictor of progression and survival in patients with uterine and ovarian carcinomas.pdf`

  - **Status:** <font color='green'>Success</font>

  - **Analysed Biomarkers:** n/a

---

In [ ]:
import csv
import pandas as pd

# Replace this with your actual list of dictionaries containing extracted data
# For example: extracted_data = [{'Title of Paper': '...', 'Author list': '...', ...}, ...]
extracted_data = all_results
csv_file_path = 'extracted_results.csv'

# Get all unique fieldnames from your data
fieldnames = list({key for d in extracted_data for key in d.keys()})

with open(csv_file_path, mode='w', newline='', encoding='utf-8') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for row in extracted_data:
        writer.writerow(row)

print(f"\nExported extracted results to {csv_file_path}")



Exported extracted results to extracted_results.csv


In [ ]:
#S1 data visualization
import pandas as pd

# Convert data to DataFrame and save to CSV
df = pd.DataFrame(extracted_data)
df.to_csv('extracted_results.csv', index=False)

# Set row numbers (index) to start at 1
df.index = range(1, len(df) + 1)

# Display all rows
pd.set_option('display.max_rows', None)
df


,pdf_uri,Analysed Biomarkers
1,gs://tlab-extractionbucket3/papers3/S023_Frequ...,n/a
2,gs://tlab-extractionbucket3/papers3/S024_Marke...,n/a
3,gs://tlab-extractionbucket3/papers3/S025_Endom...,n/a
4,gs://tlab-extractionbucket3/papers3/S026_Assoc...,n/a
5,gs://tlab-extractionbucket3/papers3/S027_Sex S...,n/a
6,gs://tlab-extractionbucket3/papers3/S028_PIK3C...,n/a
7,gs://tlab-extractionbucket3/papers3/S029_MMR d...,n/a
8,gs://tlab-extractionbucket3/papers3/S100_Funct...,n/a
9,gs://tlab-extractionbucket3/papers3/S101_Ident...,n/a
10,gs://tlab-extractionbucket3/papers3/S102_L1CAM...,n/a
